In [12]:
#TODO make sure this renders in the github repo

# Data Loader

The decoder-only model (Llama 3 8B), is trained on next-token prediction using a causal mask: Given a continuous stream of tokens, we slice a fixed-length window (e.g., 256 tokens).

- **Input** ($X$): Tokens $0$ to $N-1$, e.g., $[x_0, x_1, ..., x_{N-1}]$
- **Target** ($Y$): Tokens $1$ to $N$, e.g., $[x_1, x_2, ..., x_{N}]$
- Timestamp example sequence: "This is the Llama 3 8B"
  - The **causal mask** ensures that the tokenized representation for token $i$ is calculated using only the tokens at positions $\leq i$, this prevents the model from "cheating" by looking at the target tokens during training.
  - **Note:** The timestamp table is only for visualization purposes, in practice, the model processes the entire window in one forward pass!

    | Timestamp | Input | Target |
    | --- | --- | --- |
    | 1 | "This" | "is" |
    | 2 | "This is" | "the" |
    | 3 | "This is the" | "Llama" |
    | 4 | "This is the Llama" | "3" |
    | 5 | "This is the Llama 3" | "8B" |

**Example of what the tokens will look like:**

```text
First 10 tokens of the Input:  
    [1, 269, 72, 224, 44, 81, 71, 72, 83, 262]
First 10 tokens of the Target (shifted right): 
    [269, 72, 224, 44, 81, 71, 72, 83, 262, 71]
```

- The dataloader yields dense, packed **batches of tokens**.

In [13]:
# @i-c
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import EasyJupyter
import torch
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer

In [15]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig

In [16]:
def get_raw_dataset():
    """Used specifically when training a custom BPETokenizer(), we need to feed it raw un-tokenized text from the stream."""
    raw_dataset_stream = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True,
    )
    return raw_dataset_stream

In [17]:
class FineWebEduDataset(IterableDataset):
    def __init__(self, cfg: BaseConfig, tokenizer):
        """
        Handles loading the HuggingFace FineWeb-edu dataset.

        Args:
            tokenizer: Either my custom trained BPE tokenizer, or the HuggingFace tokenizer for the Pre-trained Llama 3 models. 
        """
        self.cfg = cfg

        # Load the a subset randomly sampled from the whole dataset of around 10-Billion gpt2 tokens in streaming mode.
        self.dataset = load_dataset(
            "HuggingFaceFW/fineweb-edu",
            name="sample-10BT",
            split="train",
            streaming=True,
        )

        self.tokenizer = tokenizer

        self.doc_end_token = self.tokenizer.token_to_id(
            cfg.special_tokens["doc_end_token"]["token"]
        )

    def __iter__(self):
        """
        Iterate through the dataset and yield tokenized input and target pairs.
        """
        buffer = []

        context_len = self.cfg.context_len # The maximum number of tokens the model can remember.

        for example in self.dataset:
            # Tokenize the raw text document.
            tokens = self.tokenizer.encode(example["text"]).ids

            # Append the document tokens and the doc_end_token separator.
            buffer.extend(tokens)
            buffer.append(self.doc_end_token)

            # Once we have enough tokens for a full context window + 1 (for the target shift)
            while len(buffer) >= context_len + 1:
                # Slice the exact context window
                chunk = buffer[: context_len + 1]

                # Shift the buffer forward so the next context window picks up where this one left off.
                buffer = buffer[context_len :]

                # Input is the first N tokens
                x = torch.tensor(chunk[:-1], dtype=torch.long)
                # Target is shifted 1 token to the right
                y = torch.tensor(chunk[1:], dtype=torch.long)

                yield x, y

In [18]:
def create_causal_dataloader(cfg: BaseConfig, tokenizer):
    """
    Creates a causal dataloader that yields dense, packed batches of tokens.

    Args:
        tokenizer: Either my custom trained BPE tokenizer, or the HuggingFace tokenizer for the Pre-trained Llama 3 models.
    """

    dataset = FineWebEduDataset(cfg=cfg, tokenizer=tokenizer)

    return DataLoader(
        dataset,
        batch_size=cfg.micro_batch_size,
        num_workers=cfg.num_workers,
    )

## Test

In [19]:
# @i-c
from llama_configs import Llama3_scaled_down
from model.tokenizer import BPETokenizer

cfg = Llama3_scaled_down()
cfg.vocab_size = 10_000

# Get a trained tokenizer
t = BPETokenizer(cfg)
success, tokenizer = t.load_tokenizer(load_test_tokenizer= True)
if not success:
    print(
        "A test tokenizer is needed, please run the test case in the tokenizer.ipynb notebook first!"
    )
    exit(1)

dataloader = create_causal_dataloader(cfg=cfg, tokenizer=tokenizer)

# Fetch a single batch
print("\n\nFetching a single batch from FineWeb-Edu...")
data_iterator = iter(dataloader)
x, y = next(data_iterator)
print("\n\nA single batch of data:")
print(f"x: {x}")
print(f"\ny: {y}")


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM

Loading existing trained BPE Tokenizer from: (/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/saved_models/universal_tokenizer/custom_BPE_tokenizer_vocab_size_10000.json)...


Fetching a single batch from FineWeb-Edu...


A single batch of data:
x: tensor([[   1,  456, 8887,  ..., 2109, 7204,  325],
        [8805,  362, 8720,  ...,  756,  294,  541],
        [1982,  751,  837,  ...,  380,   17,  202],
        ...,
        [8976,  281, 2856,  ...,  294, 3171, 6787],
        [4236,   15,  311,  ...,  672, 1558,  281],
        [ 266, 2124, 3825,  ...,  573,  420, 1879]])

y: tensor([[ 456, 8887,  303,  ..., 7204,  325, 8805],
        [ 362, 8720,   17,  ...,  294,  541, 1982],
        [ 751,  837,  443,  ...,   17,  202,   40],
        ...,
        [ 281, 2856, 2127,  ..., 3171, 6787, 4236],
        [  15,  311, 6425,  ..., 1558,  281,  266],
        [2124, 3825,   17,  ...,  420, 1879, 7082]])

In [20]:
# @i-c:
print("\n\nDecoding the first 50 tokens of x[0] to view the document:")
token_ids_list = x[0][:50].tolist()
decoded_document = tokenizer.decode(token_ids_list)
print(f"decoded_document: [{decoded_document}...]")



Decoding the first 50 tokens of x[0] to view the document:
decoded_document: [The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizab...]


In [21]:
# @i-c:
# Verify the shapes match the llama cfg
print(f"\n\nInput 'x' shape: {x.shape} -> Expected shape: (Micro Batch size, Context length)")
print(f"Input 'y' shape: {y.shape} -> Expected shape: (Micro Batch size, Context length)")



Input 'x' shape: torch.Size([8, 1024]) -> Expected shape: (Micro Batch size, Context length)
Input 'y' shape: torch.Size([8, 1024]) -> Expected shape: (Micro Batch size, Context length)


In [ ]:
# @i-c:
# Verify the target shape are shifted correctly
print(f"\n\nFirst 10 tokens of x[0]: {x[0][:10].tolist()}")
print(f"\nFirst 10 tokens of y[0]: {y[0][:10].tolist()}")



First 10 tokens of x[0]: [1, 456, 8887, 303, 2526, 202, 2776, 522, 266, 4493]

First 10 tokens of y[0]: [456, 8887, 303, 2526, 202, 2776, 522, 266, 4493, 15]


: 